# KBStats average points × odds × lineup score

Creates an optimizer-ready full KBStats score file from the newest matchday-qualified season-average-points analysis, expected match points from odds, configurable-weight LigaInsider, Kickbase, Kicker, and RotoWire starting chances. LigaInsider and Kicker alternatives use geometric per-slot decay; repeated Kickbase alternatives are pooled as team-level evidence and normalized to an expected XI while remaining below displayed starters. Kickbase and Kicker display labels resolve automatically to a unique same-team KBStats last name, then first name; shared-name or no-match candidates are prompted only at 50% similarity or higher and persist to `data/reference/kickbase_player_name_cross_references.csv` with a provider scope. A Kickbase label therefore cannot override a Kicker label for the same player, or vice versa. Players not present in the qualified analysis receive a metric and score of zero; valid negative KBStats averages are retained and can yield negative scores.

Goalkeeper expected match points are `1.1 + raw expected match points * (0.6 / 3)`: raw 0 maps to 1.1, 1.5 maps to 1.4, and 3 maps to 1.7. Outfield players retain raw 0–3 values. The exported `expected_match_points` is the value used in the score. Rerun this notebook to generate updated scores.

Goalkeeper alternatives use decay **0.60** (relative weights **1, 0.6, 0.36, …**), with existing provider normalization and injury handling. Outfield alternatives retain their configured decay.


In [ ]:
import importlib
import sys
from pathlib import Path

QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY = 0.15
# Each rank is 45% of the preceding alternative before its slot is normalized.
ALTERNATIVE_STARTING_CHANCE_DECAY = 0.45
GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY = 0.60
# Relative trust weights for active lineup sources; use 0 to disable one.
LINEUP_SOURCE_WEIGHTS = {'ligainsider': 4.0, 'kickbase': 3.0, 'kicker': 2.0, 'rotowire': 1.0}
# You will confirm whether each source is current for the selected matchday.


def locate_project_root() -> Path:
    starts = [Path.cwd().resolve()]
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.insert(0, Path(notebook_path).expanduser().resolve().parent)
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py.')


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import kbstats_points_odds_lineup_score

kbstats_points_odds_lineup_score = importlib.reload(
    kbstats_points_odds_lineup_score
)

result = kbstats_points_odds_lineup_score.run_score_creation(
    'season_average',
    questionable_injury_starting_chance_penalty=QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY,
    alternative_starting_chance_decay=ALTERNATIVE_STARTING_CHANCE_DECAY,
    goalkeeper_alternative_starting_chance_decay=GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY,
    lineup_source_weights=LINEUP_SOURCE_WEIGHTS,
)
